# 02 — METR-LA data quality and handoff

The real METR-LA file has been supplied and copied unchanged into `data/raw/metr-la.h5`; `load_context` verifies its recorded checksum before analysis. Executing these cells computes findings from that artifact. The notebook metadata records the actual interpreter, package versions and execution method. Report and metadata outputs can be regenerated from the same raw file.

Person A preserves the canonical raw file byte-for-byte. This notebook audits structure, missing and zero values, candidate outages, constant sensors and abrupt changes. It does not impute, fit forecasting transforms, remove sensors or construct Person B's features.

In [1]:
from pathlib import Path
import os
import sys
os.environ["MPLBACKEND"] = "Agg"
import matplotlib
matplotlib.use("Agg", force=True)

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(
    (candidate for candidate in candidates
     if (candidate / "src/data/run_person_a.py").is_file()
     and (candidate / "configs/data_quality.yaml").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the project root or its notebooks directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".mplconfig"))

import numpy as np
import pandas as pd
from IPython.display import Image, JSON, Markdown, display
from src.data.run_person_a import load_context, write_analysis_outputs
from src.data.quality import DEFAULT_CONFIG, analyze_frame

print("Project root:", ROOT)

Project root: /Users/pankajbaid/projects/AML/traffic-flow-capstone


In [2]:
# Verifies all recorded raw-file checksums, reads one HDF5 table and loads
# trusted copied graph metadata. Missing speed data is an explicit error.
context = load_context(ROOT)
df = context["df"]
quality_config = {
    key: value for key, value in context["config"].items()
    if key in DEFAULT_CONFIG
}
analysis = analyze_frame(
    df=df,
    locations=context["locations"],
    graph_sensor_ids=context["graph_sensor_ids"],
    adjacency=context["adjacency"],
    config=quality_config,
)
valid = analysis["valid_speed"]
summary = analysis["summary"]
print("Validation status:", summary["validation_status"])
print("Raw shape:", df.shape)
assert valid.index.equals(df.index) and valid.columns.equals(df.columns)

Validation status: warning
Raw shape: (34272, 207)


## Structural checks and quality summary

Inspect the actual dimensions, start/end, timezone metadata, cadence, duplicates and missing timestamps. The full summary is shown without replacing observed values with nominal benchmark characteristics. Structural failures require resolution before the canonical handoff; warning-level diagnostics are not automatic deletion rules.

In [3]:
display(JSON(summary, expanded=True))
display(pd.DataFrame({"issue": summary["issues"]}))

<IPython.core.display.JSON object>

,issue
0,575302 suspect/invalid speed values are masked...
1,135 positive constant runs warrant inspection;...
2,7990 large consecutive speed changes warrant i...


## Sensor identity, locations and missingness

The sensor table follows HDF5 column order. Check graph correspondence by sensor ID, not by length alone. Counts named `missing_count` refer to NaN cells; `invalid_count` also includes exact zero, negative and infinite readings. Fractions use existing timestamp rows. Absent timestamps are reported separately below.

In [4]:
display(analysis["sensors"])
display(analysis["missingness_summary"])
display(analysis["missingness_over_time"].head(24))
display(
    analysis["missingness_over_time"].sort_values(
        "invalid_fraction", ascending=False, kind="stable"
    ).head(24)
)

,sensor_id,position,observation_count,valid_count,missing_count,infinite_count,nonfinite_count,zero_count,negative_count,invalid_count,valid_fraction,valid_mean,valid_std,valid_min,valid_median,valid_max,is_constant,is_raw_constant,is_all_zero,is_low_variance,zero_run_count,zero_outage_run_count,max_zero_run,flatline_run_count,max_flatline_run,discontinuity_count,latitude,longitude
0,773869,0,34272,30429,0,0,0,3843,0,3843,0.887868,61.530971,12.316332,2.500000,65.375000,70.0,False,False,False,False,142,37,1118,0,3,77,34.15497,-118.31829
1,767541,1,34272,32123,0,0,0,2149,0,2149,0.937296,64.497026,3.355915,10.875000,65.222222,70.0,False,False,False,False,97,27,381,0,3,2,34.11621,-118.23799
2,767542,2,34272,32123,0,0,0,2149,0,2149,0.937296,64.788643,9.726387,8.111111,67.555556,70.0,False,False,False,False,97,27,381,0,3,8,34.11641,-118.23819
3,717447,3,34272,32098,0,0,0,2174,0,2174,0.936566,52.878571,9.520243,1.000000,54.333333,70.0,False,False,False,False,104,27,381,0,3,16,34.07248,-118.26772
4,717446,4,34272,31809,0,0,0,2463,0,2463,0.928134,49.647798,15.042630,3.000000,47.875000,70.0,False,False,False,False,122,30,381,0,3,21,34.07142,-118.26572
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202,717592,202,34272,31175,0,0,0,3097,0,3097,0.909635,62.569165,3.575920,2.875000,63.333333,70.0,False,False,False,False,105,30,852,0,3,9,34.14604,-118.22430
203,717595,203,34272,32008,0,0,0,2264,0,2264,0.933940,66.904365,3.500657,6.000000,67.750000,70.0,False,False,False,False,105,30,381,0,3,11,34.14163,-118.18290
204,772168,204,34272,32122,0,0,0,2150,0,2150,0.937267,58.358403,8.363177,2.000000,61.666667,70.0,False,False,False,False,98,27,381,0,4,2,34.16542,-118.47985
205,718141,205,34272,31746,0,0,0,2526,0,2526,0.926296,63.608437,9.737222,5.444444,66.444444,70.0,False,False,False,False,127,39,381,0,10,221,34.15133,-118.37456


,sensor_id,position,observation_count,valid_count,missing_count,infinite_count,nonfinite_count,zero_count,negative_count,invalid_count,missing_fraction,infinite_fraction,nonfinite_fraction,zero_fraction,negative_fraction,invalid_fraction
0,773869,0,34272,30429,0,0,0,3843,0,3843,0.0,0.0,0.0,0.112132,0.0,0.112132
1,767541,1,34272,32123,0,0,0,2149,0,2149,0.0,0.0,0.0,0.062704,0.0,0.062704
2,767542,2,34272,32123,0,0,0,2149,0,2149,0.0,0.0,0.0,0.062704,0.0,0.062704
3,717447,3,34272,32098,0,0,0,2174,0,2174,0.0,0.0,0.0,0.063434,0.0,0.063434
4,717446,4,34272,31809,0,0,0,2463,0,2463,0.0,0.0,0.0,0.071866,0.0,0.071866
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202,717592,202,34272,31175,0,0,0,3097,0,3097,0.0,0.0,0.0,0.090365,0.0,0.090365
203,717595,203,34272,32008,0,0,0,2264,0,2264,0.0,0.0,0.0,0.066060,0.0,0.066060
204,772168,204,34272,32122,0,0,0,2150,0,2150,0.0,0.0,0.0,0.062733,0.0,0.062733
205,718141,205,34272,31746,0,0,0,2526,0,2526,0.0,0.0,0.0,0.073704,0.0,0.073704


,missing_count,zero_count,invalid_count,valid_count,missing_fraction,zero_fraction,invalid_fraction,all_zero,all_invalid
2012-03-01 00:00:00,0,2,2,205,0.0,0.009662,0.009662,False,False
2012-03-01 00:05:00,0,2,2,205,0.0,0.009662,0.009662,False,False
2012-03-01 00:10:00,0,2,2,205,0.0,0.009662,0.009662,False,False
2012-03-01 00:15:00,0,207,207,0,0.0,1.000000,1.000000,True,True
2012-03-01 00:20:00,0,207,207,0,0.0,1.000000,1.000000,True,True
2012-03-01 00:25:00,0,2,2,205,0.0,0.009662,0.009662,False,False
2012-03-01 00:30:00,0,2,2,205,0.0,0.009662,0.009662,False,False
2012-03-01 00:35:00,0,2,2,205,0.0,0.009662,0.009662,False,False
2012-03-01 00:40:00,0,2,2,205,0.0,0.009662,0.009662,False,False
2012-03-01 00:45:00,0,2,2,205,0.0,0.009662,0.009662,False,False


,missing_count,zero_count,invalid_count,valid_count,missing_fraction,zero_fraction,invalid_fraction,all_zero,all_invalid
2012-03-01 00:15:00,0,207,207,0,0.0,1.0,1.0,True,True
2012-03-01 00:20:00,0,207,207,0,0.0,1.0,1.0,True,True
2012-03-03 00:15:00,0,207,207,0,0.0,1.0,1.0,True,True
2012-03-05 13:30:00,0,207,207,0,0.0,1.0,1.0,True,True
2012-03-05 13:35:00,0,207,207,0,0.0,1.0,1.0,True,True
2012-03-05 13:40:00,0,207,207,0,0.0,1.0,1.0,True,True
2012-03-05 13:45:00,0,207,207,0,0.0,1.0,1.0,True,True
2012-03-05 13:50:00,0,207,207,0,0.0,1.0,1.0,True,True
2012-03-05 13:55:00,0,207,207,0,0.0,1.0,1.0,True,True
2012-03-05 14:00:00,0,207,207,0,0.0,1.0,1.0,True,True


## Missing timestamps and invalid sensor readings

Do not treat adjacent observed rows as five minutes apart across a gap. Person B must create the complete 5-minute grid with missing-row flags before lag and horizon construction. Duplicate or off-grid timestamps must be resolved explicitly. No rows are silently added, removed or averaged in this analysis view.

In [5]:
display(analysis["timestamp_gaps"])
timestamp_details = summary["timestamp_validation"]
display(pd.DataFrame({"missing_timestamp": timestamp_details["missing_timestamps"]}))
display(pd.DataFrame({"off_grid_timestamp": timestamp_details["off_grid_timestamps"]}))
display(pd.Series(summary["value_counts"], name="measured_cells").to_frame())

,previous_timestamp,next_timestamp,elapsed_minutes,expected_missing_slots,aligned_to_cadence


,missing_timestamp


,off_grid_timestamp


,measured_cells
missing_nan,0
infinite,0
zero,575302
negative_finite,0
valid_positive_finite,6519002
invalid_total,575302


## Investigate zeros without assigning an unsupported cause

The [official historical-average baseline](https://github.com/liyaguang/DCRNN/blob/602afd9d767d3aa1c9b3eac51710d6aeee12c227/scripts/eval_baseline_methods.py#L12-L30) excludes zero readings from its means, and [baseline evaluation](https://github.com/liyaguang/DCRNN/blob/602afd9d767d3aa1c9b3eac51710d6aeee12c227/scripts/eval_baseline_methods.py#L89-L106) masks zero targets. The [metric implementation](https://github.com/liyaguang/DCRNN/blob/602afd9d767d3aa1c9b3eac51710d6aeee12c227/lib/metrics.py#L62-L98) applies the supplied null mask.

Our predeclared rule treats exact zeros as benchmark missing/suspect, retains positive near-zero observations and preserves raw bytes. Long zero runs or simultaneous zero readings support investigation; they do not establish whether a particular zero was an outage, corruption or real stopped traffic. Run lengths break at gaps and duplicated timestamps. Duration counts occupied sampling slots; first-to-last timestamp span is one interval shorter.

In [6]:
display(JSON(summary["zero_evidence"], expanded=True))
zero_runs = analysis["zero_runs"]
display(zero_runs.head(50))
display(
    zero_runs.loc[zero_runs["possible_outage"]].sort_values(
        "length", ascending=False, kind="stable"
    ).head(50)
)

<IPython.core.display.JSON object>

,sensor_id,position,start_timestamp,end_timestamp,length,duration_minutes,possible_outage
0,773869,0,2012-03-01 00:15:00,2012-03-01 00:20:00,2,10.0,False
1,773869,0,2012-03-03 00:15:00,2012-03-03 00:15:00,1,5.0,False
2,773869,0,2012-03-03 10:30:00,2012-03-03 10:30:00,1,5.0,False
3,773869,0,2012-03-03 18:15:00,2012-03-03 18:35:00,5,25.0,False
4,773869,0,2012-03-04 11:05:00,2012-03-04 11:10:00,2,10.0,False
5,773869,0,2012-03-04 18:35:00,2012-03-04 18:45:00,3,15.0,False
6,773869,0,2012-03-04 19:00:00,2012-03-04 19:00:00,1,5.0,False
7,773869,0,2012-03-04 19:10:00,2012-03-04 19:10:00,1,5.0,False
8,773869,0,2012-03-05 13:30:00,2012-03-05 14:50:00,17,85.0,True
9,773869,0,2012-03-05 19:20:00,2012-03-05 19:50:00,7,35.0,False


,sensor_id,position,start_timestamp,end_timestamp,length,duration_minutes,possible_outage
9291,717490,70,2012-06-05 12:25:00,2012-06-11 08:35:00,1683,8415.0,True
9393,717491,71,2012-06-05 12:25:00,2012-06-11 08:35:00,1683,8415.0,True
7921,769444,60,2012-06-21 17:35:00,2012-06-26 08:20:00,1330,6650.0,True
8407,769443,63,2012-06-21 17:35:00,2012-06-26 08:20:00,1330,6650.0,True
10516,769431,81,2012-06-21 17:35:00,2012-06-26 08:20:00,1330,6650.0,True
10636,769430,82,2012-06-21 17:35:00,2012-06-26 08:20:00,1330,6650.0,True
24462,759772,194,2012-03-09 12:45:00,2012-03-13 13:20:00,1160,5800.0,True
3281,717804,26,2012-03-09 14:55:00,2012-03-13 13:20:00,1134,5670.0,True
14537,716968,112,2012-03-09 14:55:00,2012-03-13 13:20:00,1134,5670.0,True
18053,718496,140,2012-03-09 14:55:00,2012-03-13 13:20:00,1134,5670.0,True


## Constant sensors, low variance, positive flatlines and jumps

Default inspection thresholds are `std < 1` source-speed unit, zero/positive constant runs of at least 12 slots, and absolute changes of at least 30 source-speed units between consecutive valid five-minute readings. The effective settings below come from config. A flatline or abrupt change may be a real traffic pattern. Full-period flags are descriptive and never authorize sensor removal using held-out data.

In [7]:
display(pd.Series(summary["policy"], name="effective_rule").to_frame())
sensors = analysis["sensors"]
flagged = sensors[
    sensors["is_constant"] | sensors["is_raw_constant"]
    | sensors["is_low_variance"] | (sensors["valid_count"] == 0)
]
display(flagged)
display(analysis["flatline_runs"].sort_values("length", ascending=False).head(50))
display(
    analysis["discontinuities"].sort_values(
        "absolute_change", ascending=False, kind="stable"
    ).head(50)
)

,effective_rule
expected_sensors,207
frequency,0 days 00:05:00
low_variance_std_threshold,1.0
zero_run_min_steps,12
flatline_min_steps,12
discontinuity_threshold,30.0
invalid_values,"[NaN, infinity, negative, exact zero]"
positive_near_zero,retained
imputation,none
raw_modified,False


,sensor_id,position,observation_count,valid_count,missing_count,infinite_count,nonfinite_count,zero_count,negative_count,invalid_count,valid_fraction,valid_mean,valid_std,valid_min,valid_median,valid_max,is_constant,is_raw_constant,is_all_zero,is_low_variance,zero_run_count,zero_outage_run_count,max_zero_run,flatline_run_count,max_flatline_run,discontinuity_count,latitude,longitude


,sensor_id,position,start_timestamp,end_timestamp,length,duration_minutes,speed
8,718076,105,2012-03-07 09:25:00,2012-03-07 16:10:00,82,410.0,64.0
80,772669,166,2012-04-30 23:40:00,2012-05-01 04:45:00,62,310.0,70.0
93,772669,166,2012-05-14 00:30:00,2012-05-14 05:05:00,56,280.0,70.0
122,772669,166,2012-06-03 23:50:00,2012-06-04 04:25:00,56,280.0,70.0
117,772669,166,2012-05-29 00:40:00,2012-05-29 05:05:00,54,270.0,70.0
124,772669,166,2012-06-06 01:00:00,2012-06-06 05:20:00,53,265.0,70.0
94,772669,166,2012-05-15 00:50:00,2012-05-15 05:05:00,52,260.0,70.0
34,772669,166,2012-03-21 01:10:00,2012-03-21 05:25:00,52,260.0,70.0
54,772669,166,2012-04-10 00:50:00,2012-04-10 05:00:00,51,255.0,70.0
81,772669,166,2012-05-02 00:35:00,2012-05-02 04:40:00,50,250.0,70.0


,sensor_id,position,previous_timestamp,timestamp,previous_speed,speed,speed_change,absolute_change
7367,771673,196,2012-05-20 23:35:00,2012-05-20 23:40:00,70.000000,0.333333,-69.666667,69.666667
2502,717491,71,2012-03-16 02:50:00,2012-03-16 02:55:00,70.000000,0.888889,-69.111111,69.111111
2360,773880,67,2012-05-16 18:40:00,2012-05-16 18:45:00,70.000000,2.000000,-68.000000,68.000000
4833,718496,140,2012-04-17 22:00:00,2012-04-17 22:05:00,2.000000,69.500000,67.500000,67.500000
6470,716949,170,2012-04-29 03:40:00,2012-04-29 03:45:00,70.000000,3.000000,-67.000000,67.000000
3438,717488,103,2012-05-28 15:10:00,2012-05-28 15:15:00,2.000000,68.400000,66.400000,66.400000
6587,768469,175,2012-05-28 15:10:00,2012-05-28 15:15:00,3.000000,69.400000,66.400000,66.400000
3437,717488,103,2012-05-28 15:05:00,2012-05-28 15:10:00,68.222222,2.000000,-66.222222,66.222222
6548,717585,173,2012-03-07 09:25:00,2012-03-07 09:30:00,2.000000,67.888889,65.888889,65.888889
4762,716958,139,2012-03-19 14:20:00,2012-03-19 14:25:00,70.000000,4.285714,-65.714286,65.714286


## Inspect raw context around representative zero runs

The first three longest flagged zero runs are selected deterministically for inspection. Their surrounding raw observations and quality-masked views help distinguish isolated zeros from sustained missing sequences. This is descriptive evidence, not a fault classifier. No example is created if no run exists.

In [8]:
longest_runs = zero_runs.sort_values("length", ascending=False, kind="stable").head(3)
for event in longest_runs.itertuples(index=False):
    position = int(event.position)
    start = pd.Timestamp(event.start_timestamp)
    end = pd.Timestamp(event.end_timestamp)
    nearby = (df.index >= start - pd.Timedelta(minutes=15)) & (
        df.index <= end + pd.Timedelta(minutes=15)
    )
    # Bound notebook output while retaining both ends of a long event.
    event_view = pd.DataFrame({
        "raw_speed": df.iloc[:, position].loc[nearby],
        "valid_speed": valid.iloc[:, position].loc[nearby],
    })
    display(Markdown(f"### Sensor {event.sensor_id}: {start} to {end}"))
    if len(event_view) > 40:
        display(event_view.head(20))
        display(Markdown("Middle rows omitted from display; complete event is recorded in metadata."))
        display(event_view.tail(20))
    else:
        display(event_view)
if longest_runs.empty:
    display(Markdown("No exact-zero runs detected in the acquired data."))

### Sensor 717490: 2012-06-05 12:25:00 to 2012-06-11 08:35:00

,raw_speed,valid_speed
2012-06-05 12:10:00,64.375000,64.375000
2012-06-05 12:15:00,62.777778,62.777778
2012-06-05 12:20:00,61.666667,61.666667
2012-06-05 12:25:00,0.000000,NaN
2012-06-05 12:30:00,0.000000,NaN
2012-06-05 12:35:00,0.000000,NaN
2012-06-05 12:40:00,0.000000,NaN
2012-06-05 12:45:00,0.000000,NaN
2012-06-05 12:50:00,0.000000,NaN
2012-06-05 12:55:00,0.000000,NaN


Middle rows omitted from display; complete event is recorded in metadata.

,raw_speed,valid_speed
2012-06-11 07:15:00,0.000000,NaN
2012-06-11 07:20:00,0.000000,NaN
2012-06-11 07:25:00,0.000000,NaN
2012-06-11 07:30:00,0.000000,NaN
2012-06-11 07:35:00,0.000000,NaN
2012-06-11 07:40:00,0.000000,NaN
2012-06-11 07:45:00,0.000000,NaN
2012-06-11 07:50:00,0.000000,NaN
2012-06-11 07:55:00,0.000000,NaN
2012-06-11 08:00:00,0.000000,NaN


### Sensor 717491: 2012-06-05 12:25:00 to 2012-06-11 08:35:00

...

,raw_speed,valid_speed
2012-06-11 07:15:00,0.000000,NaN
2012-06-11 07:20:00,0.000000,NaN
2012-06-11 07:25:00,0.000000,NaN
2012-06-11 07:30:00,0.000000,NaN
2012-06-11 07:35:00,0.000000,NaN
2012-06-11 07:40:00,0.000000,NaN
2012-06-11 07:45:00,0.000000,NaN
2012-06-11 07:50:00,0.000000,NaN
2012-06-11 07:55:00,0.000000,NaN
2012-06-11 08:00:00,0.000000,NaN


### Sensor 769444: 2012-06-21 17:35:00 to 2012-06-26 08:20:00

,raw_speed,valid_speed
2012-06-21 17:20:00,0.000000,NaN
2012-06-21 17:25:00,0.000000,NaN
2012-06-21 17:30:00,29.666667,29.666667
2012-06-21 17:35:00,0.000000,NaN
2012-06-21 17:40:00,0.000000,NaN
2012-06-21 17:45:00,0.000000,NaN
2012-06-21 17:50:00,0.000000,NaN
2012-06-21 17:55:00,0.000000,NaN
2012-06-21 18:00:00,0.000000,NaN
2012-06-21 18:05:00,0.000000,NaN


Middle rows omitted from display; complete event is recorded in metadata.

,raw_speed,valid_speed
2012-06-26 07:00:00,0.000000,NaN
2012-06-26 07:05:00,0.000000,NaN
2012-06-26 07:10:00,0.000000,NaN
2012-06-26 07:15:00,0.000000,NaN
2012-06-26 07:20:00,0.000000,NaN
2012-06-26 07:25:00,0.000000,NaN
2012-06-26 07:30:00,0.000000,NaN
2012-06-26 07:35:00,0.000000,NaN
2012-06-26 07:40:00,0.000000,NaN
2012-06-26 07:45:00,0.000000,NaN


## Persist measured outputs and regenerate the report

`write_analysis_outputs` writes the measured sensor, missingness, gap and diagnostic CSVs and quality summary. It generates the ten descriptive EDA PNGs whenever timestamps are chronological and unique, including to inspect flagged gaps. It writes the timestamp split contract only when structural validation has not failed. The original HDF5 file remains untouched. The report retains any warnings or failures; writing it does not automatically establish successful handoff.

In [9]:
figure_paths = write_analysis_outputs(context, analysis, root=ROOT, plots=True)
print("Report:", ROOT / "reports/data_quality_report.md")
print("Validation status:", summary["validation_status"])
print("EDA figures generated:", len(figure_paths))
report_path = ROOT / "reports/data_quality_report.md"
display(Markdown(report_path.read_text()))

Report: /Users/pankajbaid/projects/AML/traffic-flow-capstone/reports/data_quality_report.md
Validation status: warning
EDA figures generated: 10


# METR-LA data quality report — Person A, Phase 1

**Validation status: WARNING**

## Measured descriptive findings

The source covers **2012-03-01T00:00:00 through 2012-06-27T23:55:00**, with 34,272 timestamps and 207 sensors. Speeds use unverified source units. Calendar groups use the stored clock; no timezone is inferred or converted.

Across **6,519,002 valid positive sensor cells**, mean speed is **58.46**, median **63.22**, and the 5th–95th percentile range is **25.62–68.89**.

Excluded readings total **575,302 / 7,094,304 (8.11%)**; exact zeros account for 575,302. Positive near-zero speeds remain observed. No imputation or trained preprocessing is performed.

Hourly pooled means are lowest at stored hour **17:00 (49.85)** and highest at **22:00 (62.96)**. Daily pooled means are lowest on **Friday (56.32)** and highest on **Sunday (63.21)**.

All temporal means pool valid sensor cells using their counts. They are descriptive associations; unavailable readings may be nonrandom, so these profiles can be biased. Counts and exclusion rates accompany the means.

### Weekday and weekend observations

| day type | Valid cells | Mean speed | Excluded (%) |
| --- | --- | --- | --- |
| Weekday | 4690209 | 57.20 | 7.44 |
| Weekend | 1828793 | 61.68 | 9.78 |

### Traffic-period observations

| period | Valid cells | Mean speed | Excluded (%) |
| --- | --- | --- | --- |
| Weekday morning peak | 574372 | 52.02 | 9.32 |
| Weekday evening peak | 597567 | 48.25 | 5.66 |
| Weekday off-peak | 3518270 | 59.57 | 7.43 |
| Weekend | 1828793 | 61.68 | 9.78 |

Peak windows are weekdays [07:00, 10:00) and [16:00, 19:00); weekday off-peak excludes both windows. Weekend observations are reported separately. Full hourly and day-of-week tables are exported alongside these tables.

### Strongest zero-value evidence

All 207 sensors are zero together in **2,148 timestamp rows (6.27%)**, forming **97** consecutive five-minute blocks. Gaps and duplicate timestamps interrupt blocks.

These simultaneous-zero rows account for **77.2874%** of all exact-zero sensor cells.

The longest simultaneous-zero block has **381 slots**, from **2012-03-26T08:25:00** through **2012-03-27T16:05:00** (1,905 minutes of five-minute slots; the elapsed first-to-last span is one slot shorter).

Sensors with the highest zero rates:

| Sensor | Zero cells | Zero (%) |
| --- | --- | --- |
| 718076 | 6890 | 20.10 |
| 716939 | 6464 | 18.86 |
| 760987 | 5532 | 16.14 |

Longest individual zero runs (slot duration includes all observed five-minute slots):

| sensor_id | position | start_timestamp | end_timestamp | length | duration_minutes | possible_outage |
| --- | --- | --- | --- | --- | --- | --- |
| 717490 | 70 | 2012-06-05 12:25:00 | 2012-06-11 08:35:00 | 1683 | 8,415.00 | True |
| 717491 | 71 | 2012-06-05 12:25:00 | 2012-06-11 08:35:00 | 1683 | 8,415.00 | True |
| 769444 | 60 | 2012-06-21 17:35:00 | 2012-06-26 08:20:00 | 1330 | 6,650.00 | True |

Simultaneous zeros and long zero runs are evidence for investigating unavailable measurements. They do not establish a physical cause or prove an outage. Raw values are preserved, exact zeros are flagged as benchmark missing/suspect, and no sensor is automatically removed.


## Measured dataset validation

All numeric findings below are computed from the checksum-recorded HDF5 file. Source values and timestamp timezone are preserved; speed units are unverified.

### Validation status

```json
"warning"
```

### Issues

```json
[
  "575302 suspect/invalid speed values are masked in valid_speed; raw values are preserved.",
  "135 positive constant runs warrant inspection; readings are retained.",
  "7990 large consecutive speed changes warrant inspection; readings are retained."
]
```

### Dimensions

```json
{
  "timestamps": 34272,
  "unique_timestamps": 34272,
  "sensors": 207,
  "observed_cells": 7094304,
  "expected_timestamps": 34272,
  "expected_grid_cells": 7094304,
  "missing_timestamp_cells": 0
}
```

### Timestamp validation

```json
{
  "start": "2012-03-01T00:00:00",
  "end": "2012-06-27T23:55:00",
  "timezone": null,
  "timezone_note": "Naive source timestamps; no timezone inferred.",
  "expected_sampling_minutes": 5.0,
  "modal_sampling_minutes": 5.0,
  "duplicate_rows": 0,
  "chronologically_ordered": true,
  "missing_timestamp_count": 0,
  "missing_timestamps": [],
  "off_grid_timestamp_count": 0,
  "off_grid_timestamps": [],
  "gap_count": 0
}
```

### Value counts

```json
{
  "missing_nan": 0,
  "infinite": 0,
  "zero": 575302,
  "negative_finite": 0,
  "valid_positive_finite": 6519002,
  "invalid_total": 575302
}
```

### Zero evidence

```json
{
  "all_zero_timestamp_rows": 2148,
  "at_least_90pct_zero_timestamp_rows": 2148,
  "all_invalid_timestamp_rows": 2148,
  "total_zero_runs": 25963,
  "possible_outage_runs": 6942,
  "maximum_zero_run_steps": 1683,
  "interpretation": "Zero is suspect, not proof of an outage or genuinely stopped traffic."
}
```

### Sensor quality

```json
{
  "no_valid_observations": 0,
  "constant_sensors": 0,
  "raw_constant_sensors": 0,
  "all_zero_sensors": 0,
  "low_variance_sensors": 0,
  "positive_flatline_runs": 135,
  "large_discontinuities": 7990
}
```

### Metadata validation

```json
{
  "provided": true,
  "sensor_count": 207,
  "duplicate_sensor_ids": 0,
  "missing_sensor_ids": [],
  "extra_sensor_ids": [],
  "all_sensors_covered": true
}
```

### Graph validation

```json
{
  "sensor_ids_provided": true,
  "adjacency_provided": true,
  "used_as_features": false,
  "sensor_count": 207,
  "duplicate_sensor_ids": 0,
  "same_sensor_set": true,
  "same_sensor_order": true,
  "missing_sensor_ids": [],
  "extra_sensor_ids": [],
  "adjacency_shape": [
    207,
    207
  ],
  "adjacency_shape_valid": true,
  "adjacency_finite": true
}
```

### Policy

```json
{
  "expected_sensors": 207,
  "frequency": "0 days 00:05:00",
  "low_variance_std_threshold": 1.0,
  "zero_run_min_steps": 12,
  "flatline_min_steps": 12,
  "discontinuity_threshold": 30.0,
  "invalid_values": [
    "NaN",
    "infinity",
    "negative",
    "exact zero"
  ],
  "positive_near_zero": "retained",
  "imputation": "none",
  "raw_modified": false,
  "valid_std_ddof": 1
}
```

### Generated at utc

```json
"2026-10-16T16:48:51.671283+00:00"
```

### Speed sha256

```json
"64784b76d6fb8ec9bff4b6decafb354da2bb37840468fdccee5044e511277c05"
```

## Zero and missing-value handling

The [upstream baseline](https://github.com/liyaguang/DCRNN/blob/602afd9d767d3aa1c9b3eac51710d6aeee12c227/scripts/eval_baseline_methods.py#L89-L106) masks zero targets and excludes zero observations from historical averages. We preserve raw bytes, treat exact zero as benchmark missing/suspect, mask NaN/nonfinite/negative values and retain positive near-zero observations. The data alone cannot establish whether any particular zero is stopped traffic, an outage or corruption.

Contiguous zero runs, simultaneous sensor failures, flatlines and large jumps provide diagnostic evidence. Run thresholds are predeclared in config. No sensor is automatically removed, and no full-period statistic is used to fit imputation or model selection. Imputation belongs to Person B and must use only past observations and parameters fitted on training timestamps. Invalid targets must remain masked rather than imputed.

## Diagnostic evidence tables (first 12 rows; full CSVs in metadata)

### Timestamp gaps

None detected.

### Zero runs

| sensor_id | position | start_timestamp | end_timestamp | length | duration_minutes | possible_outage |
| --- | --- | --- | --- | --- | --- | --- |
| 773869 | 0 | 2012-03-01 00:15:00 | 2012-03-01 00:20:00 | 2 | 10.0 | False |
| 773869 | 0 | 2012-03-03 00:15:00 | 2012-03-03 00:15:00 | 1 | 5.0 | False |
| 773869 | 0 | 2012-03-03 10:30:00 | 2012-03-03 10:30:00 | 1 | 5.0 | False |
| 773869 | 0 | 2012-03-03 18:15:00 | 2012-03-03 18:35:00 | 5 | 25.0 | False |
| 773869 | 0 | 2012-03-04 11:05:00 | 2012-03-04 11:10:00 | 2 | 10.0 | False |
| 773869 | 0 | 2012-03-04 18:35:00 | 2012-03-04 18:45:00 | 3 | 15.0 | False |
| 773869 | 0 | 2012-03-04 19:00:00 | 2012-03-04 19:00:00 | 1 | 5.0 | False |
| 773869 | 0 | 2012-03-04 19:10:00 | 2012-03-04 19:10:00 | 1 | 5.0 | False |
| 773869 | 0 | 2012-03-05 13:30:00 | 2012-03-05 14:50:00 | 17 | 85.0 | True |
| 773869 | 0 | 2012-03-05 19:20:00 | 2012-03-05 19:50:00 | 7 | 35.0 | False |
| 773869 | 0 | 2012-03-06 11:55:00 | 2012-03-06 12:00:00 | 2 | 10.0 | False |
| 773869 | 0 | 2012-03-06 14:10:00 | 2012-03-06 14:15:00 | 2 | 10.0 | False |

### Flatline runs

| sensor_id | position | start_timestamp | end_timestamp | length | duration_minutes | speed |
| --- | --- | --- | --- | --- | --- | --- |
| 717804 | 26 | 2012-03-06 12:05:00 | 2012-03-06 14:05:00 | 25 | 125.0 | 46.0 |
| 717804 | 26 | 2012-03-06 14:20:00 | 2012-03-06 15:55:00 | 20 | 100.0 | 46.0 |
| 717804 | 26 | 2012-03-06 16:10:00 | 2012-03-06 17:45:00 | 20 | 100.0 | 46.0 |
| 717804 | 26 | 2012-03-07 20:10:00 | 2012-03-07 22:50:00 | 33 | 165.0 | 52.0 |
| 774067 | 62 | 2012-04-19 02:00:00 | 2012-04-19 03:05:00 | 14 | 70.0 | 70.0 |
| 774067 | 62 | 2012-04-20 02:00:00 | 2012-04-20 03:00:00 | 13 | 65.0 | 70.0 |
| 774067 | 62 | 2012-04-27 02:15:00 | 2012-04-27 03:45:00 | 19 | 95.0 | 70.0 |
| 774067 | 62 | 2012-04-28 03:25:00 | 2012-04-28 04:40:00 | 16 | 80.0 | 70.0 |
| 718076 | 105 | 2012-03-07 09:25:00 | 2012-03-07 16:10:00 | 82 | 410.0 | 64.0 |
| 718496 | 140 | 2012-03-08 16:00:00 | 2012-03-08 18:45:00 | 34 | 170.0 | 65.0 |
| 764853 | 144 | 2012-05-02 17:05:00 | 2012-05-02 18:15:00 | 15 | 75.0 | 70.0 |
| 763995 | 149 | 2012-03-14 02:10:00 | 2012-03-14 03:20:00 | 15 | 75.0 | 70.0 |

### Discontinuities

| sensor_id | position | previous_timestamp | timestamp | previous_speed | speed | speed_change | absolute_change |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 773869 | 0 | 2012-03-01 18:05:00 | 2012-03-01 18:10:00 | 47.111111111111114 | 13.0 | -34.111111111111114 | 34.111111111111114 |
| 773869 | 0 | 2012-03-01 19:35:00 | 2012-03-01 19:40:00 | 30.875 | 66.0 | 35.125 | 35.125 |
| 773869 | 0 | 2012-03-04 10:45:00 | 2012-03-04 10:50:00 | 68.66666666666667 | 25.0 | -43.66666666666667 | 43.66666666666667 |
| 773869 | 0 | 2012-03-04 11:35:00 | 2012-03-04 11:40:00 | 28.0 | 65.0 | 37.0 | 37.0 |
| 773869 | 0 | 2012-03-07 19:30:00 | 2012-03-07 19:35:00 | 24.25 | 62.666666666666664 | 38.416666666666664 | 38.416666666666664 |
| 773869 | 0 | 2012-03-08 15:00:00 | 2012-03-08 15:05:00 | 62.75 | 32.75 | -30.0 | 30.0 |
| 773869 | 0 | 2012-03-09 20:05:00 | 2012-03-09 20:10:00 | 24.125 | 60.22222222222222 | 36.09722222222222 | 36.09722222222222 |
| 773869 | 0 | 2012-03-12 17:45:00 | 2012-03-12 17:50:00 | 60.375 | 27.333333333333332 | -33.04166666666667 | 33.04166666666667 |
| 773869 | 0 | 2012-03-12 18:00:00 | 2012-03-12 18:05:00 | 65.625 | 34.875 | -30.75 | 30.75 |
| 773869 | 0 | 2012-03-12 18:45:00 | 2012-03-12 18:50:00 | 21.5 | 63.888888888888886 | 42.388888888888886 | 42.388888888888886 |
| 773869 | 0 | 2012-03-13 17:10:00 | 2012-03-13 17:15:00 | 64.55555555555556 | 27.0 | -37.55555555555556 | 37.55555555555556 |
| 773869 | 0 | 2012-03-14 15:00:00 | 2012-03-14 15:05:00 | 37.111111111111114 | 5.625 | -31.486111111111114 | 31.486111111111114 |

## EDA figures

Descriptive full-period EDA is not an estimator fitted for forecasting. Missing/suspect speed values are excluded; observed-speed patterns can be biased by nonrandom missingness. Hours use the stored clock; peak definitions are weekdays 07:00–10:00 and 16:00–19:00, half-open intervals.

- [overall speed distribution](../figures/overall_speed_distribution.png)

- [sensor speed distribution](../figures/sensor_speed_distribution.png)

- [average speed by hour](../figures/average_speed_by_hour.png)

- [average speed by day of week](../figures/average_speed_by_day_of_week.png)

- [weekday weekend profile](../figures/weekday_weekend_profile.png)

- [morning evening congestion](../figures/morning_evening_congestion.png)

- [missingness by sensor](../figures/missingness_by_sensor.png)

- [missingness over time](../figures/missingness_over_time.png)

- [sensor variance distribution](../figures/sensor_variance_distribution.png)

- [sample sensor timeseries](../figures/sample_sensor_timeseries.png)

## Handoff and limitations

See [Person B handoff](person_b_handoff.md). Structural failures block the canonical handoff. Warning-level diagnostics require awareness, not automatic sensor deletion. `data/metadata/split_contract.json` defines timestamp boundaries only; Person B owns feature samples. No models, performance findings, graph features, SUMO or control were implemented.


## Person B handoff checks

Keep immutable raw measurements, canonical sensor order and shared split boundaries. Split the complete chronological 5-minute grid using floor70/floor10/remainder counts; all `+1/+3/+6/+12` targets of a sample must belong to one partition. Past history may use earlier observed partitions causally. Imputation and any learned statistics must use training-only fitted parameters; targets remain masked rather than imputed. Exclude neighboring sensor traffic, adjacency, coordinates and graph features from AML inputs.

Speed units and timestamp timezone remain unverified unless new source evidence establishes them. Use the same stored-clock peak definition and evaluation validity masks across the team. Read the [full handoff contract](../reports/person_b_handoff.md) before feature engineering. Person A's acceptance requires structural success, executed notebooks and reviewed outputs; the full Phase 1 additionally requires Persons B–E.